# MoP-Forge v0.46.0 L4 Efficiency Comparison

This Google Colab notebook trains experimental MoP-Forge comparison models on an NVIDIA L4 runtime using TinyStories and writes lightweight report artifacts under `reports/`.

Default runs:

- 100M Dense extended baseline
- 100M MoP Full extended baseline and warm-start source
- 100M Warm Adapter Norm/Head sparse candidate
- 100M Warm LoRA rank-8 sparse candidate

Optional runs:

- Routed FFN top-1 expert candidate
- Cached warm adapter tail candidate

The notebook creates reports from measured outputs only. It should not be used to claim MoP superiority from a single Colab run.

## 0. Runtime Notes

Use **Runtime > Change runtime type > GPU > L4** in Colab before running. The default step count is intentionally moderate so the notebook is practical on L4. For a serious version report, raise `MAX_STEPS` to `2000` and keep the same TinyStories corpus and split seed.

In [ ]:
# Experiment settings. Edit these before running the training cells.
REPO_URL = "https://github.com/NikanBHMNJ/MoP.git"
REPO_DIR = "/content/MoP"
REPORT_ID = "v0_46_0_l4_warm_sparse_comparison"

# L4-friendly default. Use 2000 for the full version-comparison profile.
MAX_STEPS = 300
EVAL_EVERY_STEPS = 50
SAVE_EVERY_STEPS = 150
TINYSTORIES_DATASET = "roneneldan/TinyStories"
TINYSTORIES_SPLIT = "train"
TINYSTORIES_TEXT_FIELD = "text"
TINYSTORIES_MAX_RECORDS = 6000
TINYSTORIES_STREAMING = False
CORPUS_PATH = "data/colab_tinystories_corpus.jsonl"
SPLIT_SEED = 42
MAX_TRAIN_EXAMPLES = 4000
MAX_EVAL_EXAMPLES = 512
RUN_GENERATION_EVAL = False
GENERATION_EVAL_EXAMPLES = 2

# Default comparison set.
RUN_DENSE = True
RUN_MOP_FULL = True
RUN_WARM_ADAPTER_NORM_HEAD = True
RUN_WARM_LORA_R8 = True

# Optional heavier or specialized paths.
RUN_ROUTED_FFN = False
RUN_CACHED_TAIL = False
CACHE_TEACHER_TOP_K = 16
CACHE_RECORDS_PER_SHARD = 512
CACHED_DISTILLATION_ENABLED = True
CACHED_DISTILLATION_WEIGHT = 0.2
CACHED_DISTILLATION_TEMPERATURE = 2.0
HARD_EXAMPLE_REPLAY = False
HARD_EXAMPLE_REPLAY_LOSS_THRESHOLD = None
HARD_EXAMPLE_REPLAY_MULTIPLIER = 2
TARGET_EVAL_LOSS = None

# Keep true for a real L4 report. Set false only for debugging config generation.
REQUIRE_CUDA = True


In [ ]:
from __future__ import annotations

import csv
import datetime as dt
import json
import os
import re
import shlex
import shutil
import subprocess
import textwrap
from pathlib import Path


def sh(command: str, *, cwd: str | Path | None = None, check: bool = True) -> str:
    print(f"\n$ {command}")
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}: {command}")
    return result.stdout


def read_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def read_json_from_command_output(output: str) -> dict:
    decoder = json.JSONDecoder()
    parsed: dict | None = None
    for match in re.finditer(r"\{", output):
        try:
            value, _ = decoder.raw_decode(output[match.start() :])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict):
            parsed = value
    if parsed is None:
        raise ValueError("Could not find a JSON object in command output")
    return parsed


def write_json(path: str | Path, data: dict) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")
    return path


def parse_key(output: str, key: str) -> str:
    match = re.search(rf"^{re.escape(key)}=(.+)$", output, flags=re.MULTILINE)
    if not match:
        raise ValueError(f"Could not find {key}=... in command output")
    return match.group(1).strip()


## 1. Clone, Install, And Check The L4 Runtime

In [ ]:
repo_dir = Path(REPO_DIR)
if not (repo_dir / ".git").exists():
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    sh(f"git clone --depth 1 {REPO_URL} {repo_dir}")
else:
    print(f"Using existing clone: {repo_dir}")

os.chdir(repo_dir)
sh("python -m pip install -q -e .[dev,gpu]")
sh("python -m pip install -q datasets")
sh("mopforge version")
sh("mopforge doctor")
sh("mopforge runtime detect")
sh("nvidia-smi", check=False)

import torch

cuda_available = torch.cuda.is_available()
print("CUDA available:", cuda_available)
if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print("CUDA device:", device_name)
    print("VRAM GB:", round(total_gb, 2))
    if "L4" not in device_name:
        print("WARNING: this notebook is tuned for L4; results may differ on this GPU.")
elif REQUIRE_CUDA:
    raise RuntimeError("CUDA is required for a real L4 efficiency report.")


## 2. Build A TinyStories Corpus

This converts `roneneldan/TinyStories` into MoP-Forge `TextCorpusRecord` JSONL. The default uses a bounded non-streaming HF slice such as `train[:6000]` because Colab can crash while cleaning up TinyStories streaming requests. The GPU data loader then uses the same corpus path and split seed for every comparison job.

In [ ]:
corpus_split_arg = TINYSTORIES_SPLIT
if not TINYSTORIES_STREAMING and "[" not in TINYSTORIES_SPLIT:
    corpus_split_arg = f"{TINYSTORIES_SPLIT}[:{TINYSTORIES_MAX_RECORDS}]"

corpus_output = sh(
    " ".join(
        [
            "python scripts/build_colab_hf_corpus.py",
            f"--dataset {shlex.quote(TINYSTORIES_DATASET)}",
            f"--split {shlex.quote(corpus_split_arg)}",
            f"--text-field {shlex.quote(TINYSTORIES_TEXT_FIELD)}",
            f"--max-records {TINYSTORIES_MAX_RECORDS}",
            f"--output {shlex.quote(CORPUS_PATH)}",
            "--streaming" if TINYSTORIES_STREAMING else "--no-streaming",
        ]
    )
)
CORPUS_SUMMARY = read_json_from_command_output(corpus_output)
print(json.dumps(CORPUS_SUMMARY, indent=2, sort_keys=True))


## 3. Generate L4-Specific Configs

The notebook writes derived configs under `configs/jobs/colab_l4_v046/`. It does not edit the source templates.

In [ ]:
CONFIG_DIR = Path("configs/jobs/colab_l4_v046")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

COMMON_OVERRIDES = {
    "device": "cuda",
    "precision": "bf16",
    "require_device_available": True,
    "max_steps": int(MAX_STEPS),
    "eval_every_steps": int(EVAL_EVERY_STEPS),
    "save_every_steps": int(SAVE_EVERY_STEPS),
    "max_train_examples": int(MAX_TRAIN_EXAMPLES),
    "max_eval_examples": int(MAX_EVAL_EXAMPLES),
    "corpus_path": CORPUS_PATH,
    "dataset_ref": None,
    "dataset_split": None,
    "dataset_split_id": None,
    "run_generation_eval": bool(RUN_GENERATION_EVAL),
    "generation_eval_examples": int(GENERATION_EVAL_EXAMPLES),
    "generation_max_new_tokens": 32,
    "output_root": "gpu_runs",
    "artifact_root": "artifacts",
}
if TARGET_EVAL_LOSS is not None:
    COMMON_OVERRIDES["target_eval_loss"] = float(TARGET_EVAL_LOSS)


def make_config(label: str, template: str, overrides: dict | None = None) -> Path:
    envelope = read_json(template)
    payload = dict(envelope["payload"])
    payload.update(COMMON_OVERRIDES)
    payload.update(overrides or {})
    payload["name"] = f"{REPORT_ID}_{label}"
    metadata = dict(payload.get("metadata") or {})
    metadata.update(
        {
            "colab_report_id": REPORT_ID,
            "colab_label": label,
            "corpus_path": CORPUS_PATH,
            "corpus_dataset": TINYSTORIES_DATASET,
            "corpus_split": TINYSTORIES_SPLIT,
            "corpus_effective_split": corpus_split_arg,
            "split_seed": SPLIT_SEED,
            "seed": int(SPLIT_SEED),
            "max_steps": int(MAX_STEPS),
            "cache_teacher_top_k": int(CACHE_TEACHER_TOP_K),
            "cache_records_per_shard": int(CACHE_RECORDS_PER_SHARD),
            "cached_distillation_enabled": bool(CACHED_DISTILLATION_ENABLED),
            "cached_distillation_weight": float(CACHED_DISTILLATION_WEIGHT),
            "cached_distillation_temperature": float(CACHED_DISTILLATION_TEMPERATURE),
            "hard_example_replay": bool(HARD_EXAMPLE_REPLAY),
            "hard_example_replay_loss_threshold": HARD_EXAMPLE_REPLAY_LOSS_THRESHOLD,
            "hard_example_replay_multiplier": int(HARD_EXAMPLE_REPLAY_MULTIPLIER),
            "target_eval_loss": TARGET_EVAL_LOSS,
            "hardware_target": "google_colab_l4",
            "mopforge_version": "0.46.0",
        }
    )
    payload["metadata"] = metadata
    output = CONFIG_DIR / f"{label}.json"
    write_json(output, {"kind": "gpu_train", "version": envelope.get("version", "1"), "metadata": envelope.get("metadata", {}), "payload": payload})
    return output


dense_config = make_config("dense_extended", "configs/jobs/100m_dense_extended_efficiency.json")
mop_full_config = make_config("mop_full_extended", "configs/jobs/100m_mop_full_extended_efficiency.json")
print("Dense config:", dense_config)
print("MoP full config:", mop_full_config)


## 4. Train Dense And Full-MoP Baselines

The full-MoP checkpoint is used as the warm-start source for sparse candidates.

In [ ]:
RUNS: dict[str, dict] = {}
OUTPUT_ROOT = Path("outputs") / REPORT_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def train_config(label: str, config_path: Path) -> dict:
    sh(f"mopforge gpu validate {config_path}")
    estimate = sh(f"mopforge gpu estimate {config_path}")
    (OUTPUT_ROOT / f"{label}_memory_estimate.json").write_text(estimate, encoding="utf-8")
    output = sh(f"mopforge gpu train {config_path} --device cuda --precision bf16")
    run_id = parse_key(output, "run_id")
    result_path = parse_key(output, "result_path")
    checkpoint_path = parse_key(output, "latest_checkpoint_path")
    record = {
        "label": label,
        "config_path": str(config_path),
        "run_id": run_id,
        "result_path": result_path,
        "checkpoint_path": checkpoint_path,
    }
    RUNS[label] = record
    write_json(OUTPUT_ROOT / "run_manifest.json", RUNS)
    return record


if RUN_DENSE:
    train_config("dense", dense_config)
if RUN_MOP_FULL:
    train_config("mop_full", mop_full_config)

RUNS


## 5. Train Warm Sparse Candidates

These jobs resume model weights only from the full-MoP baseline. They save trainable-only checkpoints where configured.

In [ ]:
if (RUN_WARM_ADAPTER_NORM_HEAD or RUN_WARM_LORA_R8 or RUN_CACHED_TAIL) and "mop_full" not in RUNS:
    raise RuntimeError("Warm sparse candidates require RUN_MOP_FULL=True or an existing mop_full entry in RUNS.")

base_mop_checkpoint = RUNS.get("mop_full", {}).get("checkpoint_path")

if RUN_WARM_ADAPTER_NORM_HEAD:
    warm_adapter_config = make_config(
        "warm_adapter_norm_head_64",
        "configs/jobs/100m_mop_warm_adapters_norm_head_64_colab_efficiency.json",
        {
            "resume_from_checkpoint": base_mop_checkpoint,
            "base_checkpoint_path": base_mop_checkpoint,
            "resume_model_only": True,
            "save_trainable_only_checkpoints": True,
        },
    )
    train_config("warm_adapter_norm_head_64", warm_adapter_config)

if RUN_WARM_LORA_R8:
    warm_lora_config = make_config(
        "warm_lora_rank8",
        "configs/jobs/100m_mop_warm_lora_deltas_efficiency.json",
        {
            "resume_from_checkpoint": base_mop_checkpoint,
            "base_checkpoint_path": base_mop_checkpoint,
            "resume_model_only": True,
            "save_trainable_only_checkpoints": True,
            "lora_rank": 8,
        },
    )
    train_config("warm_lora_rank8", warm_lora_config)

RUNS


## 6. Optional: Cached Tail And Routed FFN Experiments

Cached-tail training can reduce repeated sparse-tail sweep cost, but the activation cache can be large. Routed FFN is experimental and should be compared carefully against Dense because it changes the active compute structure.

In [ ]:
if RUN_CACHED_TAIL:
    source_config = CONFIG_DIR / "warm_adapter_norm_head_64.json"
    if not source_config.exists():
        source_config = make_config(
            "warm_adapter_norm_head_64",
            "configs/jobs/100m_mop_warm_adapters_norm_head_64_colab_efficiency.json",
            {
                "resume_from_checkpoint": base_mop_checkpoint,
                "base_checkpoint_path": base_mop_checkpoint,
                "resume_model_only": True,
                "save_trainable_only_checkpoints": True,
            },
        )
    cache_path = Path("outputs") / f"{REPORT_ID}_warm_sparse_cache_manifest.json"
    sh(
        " ".join(
            [
                "mopforge gpu cache-activations",
                str(source_config),
                f"--checkpoint {base_mop_checkpoint}",
                f"--output {cache_path}",
                "--dtype bf16",
                f"--teacher-top-k {CACHE_TEACHER_TOP_K}",
                f"--records-per-shard {CACHE_RECORDS_PER_SHARD}",
            ]
        )
    )
    cached_config = make_config(
        "cached_warm_adapter_norm_head_64",
        "configs/jobs/100m_mop_warm_adapters_norm_head_64_colab_efficiency.json",
        {
            "resume_from_checkpoint": base_mop_checkpoint,
            "base_checkpoint_path": base_mop_checkpoint,
            "resume_model_only": True,
            "save_trainable_only_checkpoints": True,
            "activation_cache_path": str(cache_path),
            "offload_frozen_backbone_for_cache": True,
            "distillation_enabled": bool(CACHED_DISTILLATION_ENABLED),
            "distillation_weight": float(CACHED_DISTILLATION_WEIGHT),
            "distillation_temperature": float(CACHED_DISTILLATION_TEMPERATURE),
            "distillation_top_k": int(CACHE_TEACHER_TOP_K),
            "hard_example_replay_enabled": bool(HARD_EXAMPLE_REPLAY),
            "hard_example_replay_loss_threshold": HARD_EXAMPLE_REPLAY_LOSS_THRESHOLD,
            "hard_example_replay_multiplier": int(HARD_EXAMPLE_REPLAY_MULTIPLIER),
        },
    )
    train_config("cached_warm_adapter_norm_head_64", cached_config)

if RUN_ROUTED_FFN:
    if "dense" not in RUNS:
        raise RuntimeError("Routed FFN warm-start expects the dense checkpoint. Set RUN_DENSE=True.")
    routed_config = make_config(
        "routed_ffn_top1",
        "configs/jobs/100m_mop_routed_ffn_expert_efficiency.json",
        {
            "resume_from_checkpoint": RUNS["dense"]["checkpoint_path"],
            "base_checkpoint_path": RUNS["dense"]["checkpoint_path"],
            "resume_model_only": True,
            "save_trainable_only_checkpoints": True,
        },
    )
    train_config("routed_ffn_top1", routed_config)

RUNS


## 7. Build Reports Under `reports/`

This cell copies only lightweight metadata artifacts. It intentionally refuses to put `.pt`, `.pth`, `.ckpt`, `.bin`, or `.safetensors` files in the report folder.

In [ ]:
REPORT_DIR = Path("reports") / REPORT_ID
REPORT_RUNS_DIR = REPORT_DIR / "runs"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_RUNS_DIR.mkdir(parents=True, exist_ok=True)

lightweight_files = [
    "config.json",
    "gpu_training_result.json",
    "metrics.json",
    "runtime.json",
    "state.json",
    "memory_estimate.json",
]


def copy_lightweight_run(label: str, run_id: str) -> None:
    src = Path("gpu_runs") / run_id
    dst = REPORT_RUNS_DIR / label
    dst.mkdir(parents=True, exist_ok=True)
    for name in lightweight_files:
        candidate = src / name
        if candidate.exists():
            shutil.copy2(candidate, dst / name)
    config_path = RUNS[label].get("config_path")
    if config_path and Path(config_path).exists():
        shutil.copy2(config_path, dst / "launched_config.json")


for label, record in RUNS.items():
    copy_lightweight_run(label, record["run_id"])

write_json(REPORT_DIR / "run_manifest.json", RUNS)
write_json(
    REPORT_DIR / "experiment_settings.json",
    {
        "report_id": REPORT_ID,
        "mopforge_version": "0.46.0",
        "created_utc": dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z",
        "corpus_path": CORPUS_PATH,
        "corpus_dataset": TINYSTORIES_DATASET,
        "corpus_split": TINYSTORIES_SPLIT,
        "corpus_effective_split": corpus_split_arg,
        "corpus_text_field": TINYSTORIES_TEXT_FIELD,
        "corpus_max_records": TINYSTORIES_MAX_RECORDS,
        "corpus_streaming": TINYSTORIES_STREAMING,
        "corpus_summary": CORPUS_SUMMARY,
        "split_seed": SPLIT_SEED,
        "max_steps": MAX_STEPS,
        "eval_every_steps": EVAL_EVERY_STEPS,
        "save_every_steps": SAVE_EVERY_STEPS,
        "run_generation_eval": RUN_GENERATION_EVAL,
        "cache_teacher_top_k": CACHE_TEACHER_TOP_K,
        "cache_records_per_shard": CACHE_RECORDS_PER_SHARD,
        "cached_distillation_enabled": CACHED_DISTILLATION_ENABLED,
        "cached_distillation_weight": CACHED_DISTILLATION_WEIGHT,
        "cached_distillation_temperature": CACHED_DISTILLATION_TEMPERATURE,
        "hard_example_replay": HARD_EXAMPLE_REPLAY,
        "hard_example_replay_loss_threshold": HARD_EXAMPLE_REPLAY_LOSS_THRESHOLD,
        "hard_example_replay_multiplier": HARD_EXAMPLE_REPLAY_MULTIPLIER,
        "target_eval_loss": TARGET_EVAL_LOSS,
        "hardware_target": "google_colab_l4",
    },
)

run_ids = [record["run_id"] for record in RUNS.values()]
if len(run_ids) >= 2:
    sh(
        " ".join(
            [
                "mopforge gpu compare-runs",
                *run_ids,
                f"--output {REPORT_DIR / 'comparison.json'}",
                f"--output-csv {REPORT_DIR / 'comparison.csv'}",
            ]
        )
    )

if "dense" in RUNS and RUN_GENERATION_EVAL:
    gates_dir = REPORT_DIR / "gates"
    gates_dir.mkdir(parents=True, exist_ok=True)
    for label, record in RUNS.items():
        if label in {"dense", "mop_full"}:
            continue
        sh(
            " ".join(
                [
                    "mopforge gpu gate-efficiency",
                    f"--dense-run {RUNS['dense']['run_id']}",
                    f"--sparse-run {record['run_id']}",
                    f"--output {gates_dir / (label + '_gate.json')}",
                ]
            ),
            check=False,
        )
elif "dense" in RUNS:
    gates_dir = REPORT_DIR / "gates"
    gates_dir.mkdir(parents=True, exist_ok=True)
    (gates_dir / "README.md").write_text(
        "GPU efficiency gates were not run because this TinyStories corpus workflow disables generated-code evaluation by default. Use comparison.json/csv for measured loss, throughput, VRAM, trainable ratio, active ratio, and checkpoint-size comparisons.\n",
        encoding="utf-8",
    )

bad = []
for pattern in ("*.pt", "*.pth", "*.ckpt", "*.bin", "*.safetensors"):
    bad.extend(REPORT_DIR.rglob(pattern))
if bad:
    raise RuntimeError("Report folder contains model artifacts: " + ", ".join(map(str, bad)))

print("Report directory:", REPORT_DIR)


## 8. Generate The Report README

In [ ]:
comparison_path = REPORT_DIR / "comparison.json"
rows = []
if comparison_path.exists():
    rows = read_json(comparison_path).get("runs", [])


def cell(value):
    if value is None:
        return ""
    if isinstance(value, float):
        return f"{value:.4f}"
    return str(value)


table_lines = [
    "| Run | Train loss | Eval loss | Best eval loss | Time-to-target sec | Tokens-to-target | Tokens/sec | Peak allocated VRAM | Peak reserved VRAM | Target peak reserved VRAM | Trainable ratio | Active trainable ratio | Checkpoint MB | Distill top-k | Offloaded params | Hard replay | Hard replayed | Exact match | Verifier pass | Syntax pass | Device |",
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | --- |",
]
for row in rows:
    table_lines.append(
        "| "
        + " | ".join(
            [
                cell(row.get("run_id")),
                cell(row.get("train_loss")),
                cell(row.get("eval_loss")),
                cell(row.get("best_eval_loss")),
                cell(row.get("time_to_target_loss_sec")),
                cell(row.get("tokens_to_target_loss")),
                cell(row.get("tokens_per_sec")),
                cell(row.get("peak_allocated_gb")),
                cell(row.get("peak_reserved_gb")),
                cell(row.get("target_peak_reserved_gb")),
                cell(row.get("trainable_param_ratio")),
                cell(row.get("active_trainable_param_ratio")),
                cell(row.get("checkpoint_size_mb")),
                cell(row.get("distillation_top_k")),
                cell(row.get("cached_backbone_offloaded_param_count")),
                cell(row.get("hard_example_replay_enabled")),
                cell(row.get("hard_replayed_example_count")),
                cell(row.get("gen_exact_match_rate")),
                cell(row.get("gen_verifier_pass_rate")),
                cell(row.get("gen_syntax_pass_rate")),
                cell(row.get("selected_device")),
            ]
        )
        + " |"
    )

readme = f"""# MoP-Forge v0.46.0 L4 Warm Sparse Comparison

This report was generated by `notebooks/colab_l4_v046_efficiency_comparison.ipynb`.

## Purpose

Compare Dense, full-MoP, and warm sparse MoP candidates on a Google Colab L4 runtime using the same TinyStories corpus and split seed.

## Scope

- MoP-Forge version: `0.46.0`
- Target hardware: Google Colab L4
- Corpus dataset: `{TINYSTORIES_DATASET}`
- Corpus split: `{TINYSTORIES_SPLIT}`
- Effective corpus split: `{corpus_split_arg}`
- Corpus path: `{CORPUS_PATH}`
- Corpus records: `{CORPUS_SUMMARY.get('records_written')}`
- Split seed: `{SPLIT_SEED}`
- Max steps: `{MAX_STEPS}`
- Generated-code evaluation enabled: `{RUN_GENERATION_EVAL}`
- Cached teacher top-k: `{CACHE_TEACHER_TOP_K}`
- Cache records per shard: `{CACHE_RECORDS_PER_SHARD}`
- Cached distillation enabled: `{CACHED_DISTILLATION_ENABLED}`
- Hard-example replay: `{HARD_EXAMPLE_REPLAY}`
- Hard-example replay threshold: `{HARD_EXAMPLE_REPLAY_LOSS_THRESHOLD}`
- Hard-example replay multiplier: `{HARD_EXAMPLE_REPLAY_MULTIPLIER}`
- Target eval loss: `{TARGET_EVAL_LOSS}`

## Runs

```json
{json.dumps(RUNS, indent=2, sort_keys=True)}
```

## Results

{chr(10).join(table_lines)}

## Artifacts

- `comparison.json`
- `comparison.csv`
- `run_manifest.json`
- `experiment_settings.json`
- `runs/<label>/` lightweight run metadata
- `gates/` gate reports when generated-code evaluation is enabled, otherwise a note explaining why gate reports are skipped

Checkpoint and model weight files are intentionally excluded from this report folder.

## Interpretation Rules

This report supports only measured claims. On TinyStories, a sparse run is not a same-quality efficiency win unless it remains close to Dense eval loss while improving a named efficiency axis such as VRAM, trainable parameters, checkpoint size, cached-tail training time, teacher-distilled time-to-target-loss, or active expert compute. The next serious report should use a fixed code dataset and include verifier/syntax quality checks.

## Limitations

- One Colab run is not a paper-grade result.
- TinyStories is a small story-style language-modeling corpus, not a coding benchmark.
- L4 availability, thermal behavior, and Colab scheduling can change throughput.
- Short `MAX_STEPS` values are useful for workflow validation but not final quality claims.
- CPU fallback results must not be interpreted as GPU efficiency evidence.
"""

(REPORT_DIR / "README.md").write_text(readme, encoding="utf-8")
print((REPORT_DIR / "README.md").read_text(encoding="utf-8")[:4000])


## 9. Download Or Copy The Report

The report folder is lightweight. It does not include checkpoints. Use the zip for a local backup, or commit the report folder after reviewing it.

In [ ]:
zip_path = Path("reports") / f"{REPORT_ID}.zip"
if zip_path.exists():
    zip_path.unlink()
sh(f"zip -r {zip_path} {REPORT_DIR} -x '*.pt' '*.pth' '*.ckpt' '*.bin' '*.safetensors' '*/checkpoints/*'")
print("Report folder:", REPORT_DIR)
print("Report zip:", zip_path)

try:
    from google.colab import files

    files.download(str(zip_path))
except Exception as exc:
    print("Download helper unavailable outside Colab:", exc)
